# Phase 1 — Exploratory Data Analysis (EDA)
**Dataset**: doevent/perfume — 26k+ perfumes

Goals:
- Understand the shape and quality of the data
- Distribution of scent families, gender, top brands
- Top ingredients / notes
- Missing value analysis

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from data_loader import load_data

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('pastel')

df = load_data()
print(f'Total perfumes: {len(df):,}')
df.head(3)

## 1. Dataset Shape & Column Overview

In [ ]:
print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nSample row:')
df.iloc[0].to_dict()

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0]
print('Columns with missing values:')
print(missing_df)

# Check empty strings and empty lists
print('\nEmpty ingredients lists:', (df['ingredients'].apply(lambda x: len(x) == 0 if isinstance(x, list) else True)).sum())
print('Empty family:', (df['family'].isna() | (df['family'] == '')).sum())
print('Empty subfamily:', (df['subfamily'].isna() | (df['subfamily'] == '')).sum())
print('Empty gender:', (df['gender'].isna() | (df['gender'] == '')).sum())

## 3. Gender Distribution

In [ ]:
gender_counts = df['gender'].fillna('Unknown').value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(gender_counts.index, gender_counts.values, color=sns.color_palette('pastel')[:len(gender_counts)])
ax1.set_title('Perfume Count by Gender', fontweight='bold')
ax1.set_xlabel('Gender')
ax1.set_ylabel('Count')
for i, (v, k) in enumerate(zip(gender_counts.values, gender_counts.index)):
    ax1.text(i, v + 50, f'{v:,}', ha='center', fontsize=10)

ax2.pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
        colors=sns.color_palette('pastel')[:len(gender_counts)], startangle=90)
ax2.set_title('Gender Split (%)', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/eda_gender.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Scent Family Distribution

In [ ]:
family_counts = df['family'].fillna('Unknown').value_counts().head(20)

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(family_counts.index[::-1], family_counts.values[::-1],
               color=sns.color_palette('muted', len(family_counts)))
ax.set_title('Top 20 Scent Families', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Perfumes')
for bar, val in zip(bars, family_counts.values[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../data/eda_family.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Top 10 Subfamilies

In [ ]:
subfamily_counts = df['subfamily'].fillna('Unknown').value_counts().head(20)

fig, ax = plt.subplots(figsize=(13, 6))
ax.barh(subfamily_counts.index[::-1], subfamily_counts.values[::-1],
        color=sns.color_palette('pastel', len(subfamily_counts)))
ax.set_title('Top 20 Scent Subfamilies', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Perfumes')
plt.tight_layout()
plt.savefig('../data/eda_subfamily.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Top 30 Brands

In [ ]:
brand_counts = df['brand'].fillna('Unknown').value_counts().head(30)

fig, ax = plt.subplots(figsize=(13, 8))
ax.barh(brand_counts.index[::-1], brand_counts.values[::-1],
        color=sns.color_palette('coolwarm', len(brand_counts)))
ax.set_title('Top 30 Brands by Number of Perfumes', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Perfumes')
plt.tight_layout()
plt.savefig('../data/eda_brands.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Total unique brands: {df["brand"].nunique():,}')

## 7. Top 40 Ingredients / Notes

In [ ]:
all_ingredients = []
for ing_list in df['ingredients'].dropna():
    if isinstance(ing_list, list):
        all_ingredients.extend([i.strip() for i in ing_list if i])

ingredient_counts = Counter(all_ingredients)
top_ingredients = pd.DataFrame(ingredient_counts.most_common(40),
                                columns=['ingredient', 'count'])

print(f'Total unique ingredients: {len(ingredient_counts):,}')
print(f'Total ingredient mentions: {sum(ingredient_counts.values()):,}')

fig, ax = plt.subplots(figsize=(13, 10))
ax.barh(top_ingredients['ingredient'][::-1], top_ingredients['count'][::-1],
        color=sns.color_palette('flare', 40))
ax.set_title('Top 40 Most Common Ingredients / Notes', fontweight='bold', fontsize=13)
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig('../data/eda_ingredients.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Ingredients per Perfume Distribution

In [ ]:
df['n_ingredients'] = df['ingredients'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df['n_ingredients'], bins=30, color='steelblue', edgecolor='white')
ax.set_title('Distribution of Ingredients per Perfume', fontweight='bold')
ax.set_xlabel('Number of Ingredients')
ax.set_ylabel('Count')
ax.axvline(df['n_ingredients'].median(), color='tomato', linestyle='--',
           label=f"Median: {df['n_ingredients'].median():.0f}")
ax.legend()
plt.tight_layout()
plt.savefig('../data/eda_n_ingredients.png', dpi=100, bbox_inches='tight')
plt.show()

print(df['n_ingredients'].describe())

## 9. Family × Gender Heatmap

In [ ]:
top_families = df['family'].value_counts().head(12).index
df_top = df[df['family'].isin(top_families)]
pivot = df_top.pivot_table(index='family', columns='gender', aggfunc='size', fill_value=0)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Scent Family × Gender Heatmap', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../data/eda_family_gender_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. EDA Summary

In [ ]:
print('=== EDA SUMMARY ===')
print(f'Total perfumes        : {len(df):,}')
print(f'Unique brands         : {df["brand"].nunique():,}')
print(f'Unique families       : {df["family"].nunique()}')
print(f'Unique subfamilies    : {df["subfamily"].nunique()}')
print(f'Unique ingredients    : {len(ingredient_counts):,}')
print(f'Gender values         : {df["gender"].unique().tolist()}')
print(f'Avg ingredients/perfume: {df["n_ingredients"].mean():.1f}')
print(f'Perfumes with 0 ingred: {(df["n_ingredients"]==0).sum()}')